In [1]:
import pandas as pd
import numpy as np
import urllib.request
import timeit
from scipy.stats import pearsonr, spearmanr
from sklearn.preprocessing import MinMaxScaler, StandardScaler


provinces = {1: "Вінницька", 2: "Волинська", 3: "Дніпропетровська", 4: "Донецька", 5: "Житомирська"}

def download_vhi_data():
    all_dfs = []
  
    url = "https://www.star.nesdis.noaa.gov/smcd/emb/vci/VH/get_TS_admin.php?country=UKR&provinceID=1&year1=1981&year2=2024&type=Mean"
    try:
        response = urllib.request.urlopen(url)
        content = response.read().decode('utf-8')
        lines = content.split('\n')
        rows = [line.split(',') for line in lines if not line.startswith('<') and len(line) > 10]
        
        temp_df = pd.DataFrame(rows).iloc[:, :7]
        temp_df.columns = ['Year', 'Week', 'SMN', 'SMT', 'VCI', 'TCI', 'VHI']
        temp_df['Area_ID'] = 1
        temp_df['Area_Name'] = provinces[1]
        all_dfs.append(temp_df)
    except Exception as e:
        print(f"Помилка завантаження: {e}")
            
    df = pd.concat(all_dfs).dropna().reset_index(drop=True)
    for col in ['Year', 'Week', 'SMN', 'SMT', 'VCI', 'TCI', 'VHI']:
        df[col] = pd.to_numeric(df[col], errors='coerce')
    return df.dropna().reset_index(drop=True)

df_vhi = download_vhi_data()
print(f"--- Частина 1: Дані VHI для Вінниччини успішно завантажено ({len(df_vhi)} рядків) ---")


print("\n1. Вибірка ряду VHI за 2020 рік:")
t_vhi1 = timeit.timeit(lambda: df_vhi[df_vhi['Year'] == 2020][['Week', 'VHI']], number=1)
print(f"Час виконання: {t_vhi1:.5f} сек")
display(df_vhi[df_vhi['Year'] == 2020][['Week', 'VHI']].head(3))

print("\n2. Екстремуми VHI за 2022 рік:")
filtered_2022 = df_vhi[df_vhi['Year'] == 2022]
print(f"Мінімум VHI: {filtered_2022['VHI'].min()}, Максимум VHI: {filtered_2022['VHI'].max()}")

--- Частина 1: Дані VHI для Вінниччини успішно завантажено (2235 рядків) ---

1. Вибірка ряду VHI за 2020 рік:
Час виконання: 0.00100 сек


,Week,VHI
1975,1.0,37.80
1976,2.0,39.94
1977,3.0,41.75



2. Екстремуми VHI за 2022 рік:
Мінімум VHI: 36.12, Максимум VHI: 72.32


In [2]:
print("--- Частина 2: Завантаження та структурування великого датасету ---")

df_power = pd.read_csv("household_power_consumption.txt", sep=';', low_memory=False, na_values=['?'])
df_power = df_power.dropna().reset_index(drop=True)


float_cols = ['Global_active_power', 'Global_reactive_power', 'Voltage', 'Global_intensity', 'Sub_metering_1', 'Sub_metering_2', 'Sub_metering_3']
df_power[float_cols] = df_power[float_cols].astype(float)
df_power['DateTime'] = pd.to_datetime(df_power['Date'] + ' ' + df_power['Time'], dayfirst=True)

print(f"Очищено від порожніх рядків. Всього чистих записів: {len(df_power)}")


print("\n2.1 Фільтрація: Потужність > 5 кВт")
t_p1 = timeit.timeit(lambda: df_power[df_power['Global_active_power'] > 5.0], number=1)
print(f"Час: {t_p1:.5f} сек. Знайдено рядків: {len(df_power[df_power['Global_active_power'] > 5.0])}")

print("\n2.2 Фільтрація: Напруга > 235 В")
t_p2 = timeit.timeit(lambda: df_power[df_power['Voltage'] > 235.0], number=1)
print(f"Час: {t_p2:.5f} сек. Знайдено рядків: {len(df_power[df_power['Voltage'] > 235.0])}")

print("\n2.3 Фільтрація: Струм 19-20 А, Пральна машина > Бойлер")
def query_3(df): return df[(df['Global_intensity'] >= 19.0) & (df['Global_intensity'] <= 20.0) & (df['Sub_metering_2'] > df['Sub_metering_3'])]
t_p3 = timeit.timeit(lambda: query_3(df_power), number=1)
print(f"Час: {t_p3:.5f} сек. Знайдено рядків: {len(query_3(df_power))}")

print("\n2.4 Випадкова вибірка 500,000 рядків та середні значення:")
df_sample = df_power.sample(n=500000, random_state=42).reset_index(drop=True)
print(f"Середнє Кухня: {df_sample['Sub_metering_1'].mean():.2f} Вт-год, Пральня: {df_sample['Sub_metering_2'].mean():.2f} Вт-год")

print("\n2.5 Крокова вибірка після 18:00 (Кожен 3-й та 4-й рядок):")
df_evening = df_power[(df_power['DateTime'].dt.hour >= 18) & (df_power['Global_active_power'] > 6.0)].copy()
half = len(df_evening) // 2
final_split = pd.concat([df_evening.iloc[:half].iloc[::3], df_evening.iloc[half:].iloc[::4]])
print(f"Отримано рядків: {len(final_split)}")

--- Частина 2: Завантаження та структурування великого датасету ---
Очищено від порожніх рядків. Всього чистих записів: 2049280

2.1 Фільтрація: Потужність > 5 кВт
Час: 0.00567 сек. Знайдено рядків: 17547

2.2 Фільтрація: Напруга > 235 В
Час: 0.18323 сек. Знайдено рядків: 1952491

2.3 Фільтрація: Струм 19-20 А, Пральна машина > Бойлер
Час: 0.00932 сек. Знайдено рядків: 2509

2.4 Випадкова вибірка 500,000 рядків та середні значення:
Середнє Кухня: 1.12 Вт-год, Пральня: 1.31 Вт-год

2.5 Крокова вибірка після 18:00 (Кожен 3-й та 4-й рядок):
Отримано рядків: 842


In [3]:
print("--- Додаткова математична обробка (Вимога методички) ---")


df_power['Normalized_Active_Power'] = MinMaxScaler().fit_transform(df_power[['Global_active_power']])
df_power['Scaled_Voltage'] = StandardScaler().fit_transform(df_power[['Voltage']])


p_val, _ = pearsonr(df_power['Global_active_power'], df_power['Global_intensity'])
s_val, _ = spearmanr(df_power['Global_active_power'], df_power['Global_intensity'])

print(f"Коефіцієнт кореляції Пірсона (Лінійна): {p_val:.5f}")
print(f"Коефіцієнт кореляції Спірмена (Рангова): {s_val:.5f}")
print("\nУсі обчислення виконано успішно. Робота готова до збереження!")

--- Додаткова математична обробка (Вимога методички) ---
Коефіцієнт кореляції Пірсона (Лінійна): 0.99889
Коефіцієнт кореляції Спірмена (Рангова): 0.99537

Усі обчислення виконано успішно. Робота готова до збереження!
